In [4]:
# =============================================================================
# Import libraries
# =============================================================================

import pandas as pd
import numpy as np

# =============================================================================
# Load dataset
# =============================================================================

# Load the compressed detailed listings dataset
listings = pd.read_csv("../data/raw/listings.csv.gz")

print(f"Dataset shape: {listings.shape}")
listings.head()

Dataset shape: (30259, 90)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,2539,https://www.airbnb.com/rooms/2539,20260614073253,2026-06-15,city scrape,11 Min to Manhattan • Prospect Park • Fast WiFi,"Bright, serene room in a renovated apartment h...",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,2787,...,5.00,4.78,4.78,NaN,NaN,5,0,5,0,0.08
1,6848,https://www.airbnb.com/rooms/6848,20260614073253,2026-06-14,city scrape,Only 2 stops to Manhattan studio,Comfortable studio apartment with super comfor...,NaN,https://a0.muscache.com/pictures/e4f031a7-f146...,15991,...,4.80,4.69,4.59,NaN,NaN,1,1,0,0,0.95
2,6872,https://www.airbnb.com/rooms/6872,20260614073253,2026-06-14,city scrape,Uptown Sanctuary w/ Private Bath (Month to Month),This charming distancing-friendly month-to-mon...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,16104,...,5.00,5.00,5.00,NaN,NaN,2,0,2,0,0.04
3,6990,https://www.airbnb.com/rooms/6990,20260614073253,2026-06-14,city scrape,UES Beautiful Blue Room,Beautiful peaceful healthy home,NaN,https://a0.muscache.com/pictures/45fb4ec7-6856...,16800,...,4.94,4.85,4.84,NaN,NaN,1,0,1,0,1.24
4,7097,https://www.airbnb.com/rooms/7097,20260614073253,2026-06-14,city scrape,"Perfect for Your Parents, With Garden & Patio",Parents/grandparents coming to town or are you...,NaN,https://a0.muscache.com/pictures/aaac19fc-4b4d...,17571,...,4.93,4.95,4.82,NaN,NaN,2,0,2,0,2.12


In [2]:
import os

print(os.getcwd())

/Users/viviannebrown/Documents/airbnb-revenue-analysis/notebooks


In [3]:
import os

print(os.listdir("../data"))

['.DS_Store', 'processed', 'raw']


In [5]:
# Count missing values in every column

missing_values = (
    listings.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_values.head(20)

neighbourhood                  30259
instant_bookable               30259
host_neighbourhood             30259
host_thumbnail_url             30259
host_acceptance_rate           30259
host_response_rate             30259
host_response_time             30259
host_verifications             30259
host_since                     30259
host_total_listings_count      30259
neighborhood_overview          30259
calendar_updated               30259
license                        24973
host_about                     12410
bedrooms                       10973
bathrooms                      10112
beds                            9420
price_quote_total_price         8745
price_quote_price_per_night     8745
price                           8744
dtype: int64

In [6]:
# ============================================================
# Convert the price column from text to numeric
# Remove "$" and "," characters and store the cleaned values
# in a new column called price_clean
# ============================================================

listings["price_clean"] = pd.to_numeric(
    listings["price"]
        .astype("string")
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False),
    errors="coerce"
)
# Compare the original and cleaned price columns
listings[["price", "price_clean"]].head()

,price,price_clean
0,$113.97,113.97
1,$117.27,117.27
2,$80.06,80.06
3,$77.17,77.17
4,$202.47,202.47


In [7]:
# ============================================================
# Select the most important variables for cleaning and analysis
#
# These columns contain information about:
# - Price
# - Property characteristics
# - Capacity
# - Reviews
# - Availability
# - Location
# ============================================================
important_columns = [
    "price_clean",
    "room_type",
    "property_type",
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms_text",
    "review_scores_rating",
    "number_of_reviews",
    "availability_365",
    "host_is_superhost",
    "neighbourhood_cleansed"
]

listings[important_columns].head()

,price_clean,room_type,property_type,accommodates,bedrooms,beds,bathrooms_text,review_scores_rating,number_of_reviews,availability_365,host_is_superhost,neighbourhood_cleansed
0,113.97,Private room,Private room in condo,2,1.0,2.0,1 shared bath,4.80,10,339,f,Kensington
1,117.27,Entire home/apt,Entire rental unit,3,2.0,1.0,1 bath,4.60,198,191,t,Williamsburg
2,80.06,Private room,Private room in condo,1,NaN,1.0,1 shared bath,5.00,2,286,f,East Harlem
3,77.17,Private room,Private room in rental unit,1,NaN,1.0,1 shared bath,4.88,251,218,t,East Harlem
4,202.47,Private room,Private room in guest suite,2,NaN,2.0,1 private bath,4.89,423,106,t,Fort Greene


In [8]:
# ============================================================
# Count missing values in the important analysis columns
# ============================================================

important_missing = (
    listings[important_columns]
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

important_missing

bedrooms                  10973
beds                       9420
price_clean                8744
review_scores_rating       8559
host_is_superhost           349
bathrooms_text              159
room_type                     0
property_type                 0
accommodates                  0
number_of_reviews             0
availability_365              0
neighbourhood_cleansed        0
dtype: int64

In [9]:
# ============================================================
# Calculate the percentage of missing values
# ============================================================

important_missing_percent = (
    listings[important_columns]
    .isnull()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

important_missing_percent

bedrooms                  36.26
beds                      31.13
price_clean               28.90
review_scores_rating      28.29
host_is_superhost          1.15
bathrooms_text             0.53
room_type                  0.00
property_type              0.00
accommodates               0.00
number_of_reviews          0.00
availability_365           0.00
neighbourhood_cleansed     0.00
dtype: float64

In [10]:
# ============================================================
# Inspect listings with missing bedroom values
# ============================================================

listings.loc[
    listings["bedrooms"].isna(),
    [
        "property_type",
        "room_type",
        "accommodates",
        "bedrooms",
        "beds"
    ]
].head(20)

,property_type,room_type,accommodates,bedrooms,beds
2,Private room in condo,Private room,1,NaN,1.0
3,Private room in rental unit,Private room,1,NaN,1.0
4,Private room in guest suite,Private room,2,NaN,2.0
5,Entire place,Entire home/apt,2,NaN,1.0
9,Entire rental unit,Entire home/apt,2,NaN,NaN
13,Private room in rental unit,Private room,1,NaN,NaN
14,Private room in rental unit,Private room,1,NaN,NaN
16,Entire rental unit,Entire home/apt,2,NaN,1.0
17,Private room in rental unit,Private room,1,NaN,1.0
19,Entire rental unit,Entire home/apt,2,NaN,2.0


In [11]:
# ============================================================
# Inspect listings with missing bed values
# ============================================================

listings.loc[
    listings["beds"].isna(),
    [
        "property_type",
        "room_type",
        "accommodates",
        "bedrooms",
        "beds"
    ]
].head(20)

,property_type,room_type,accommodates,bedrooms,beds
8,Entire condo,Entire home/apt,2,1.0,NaN
9,Entire rental unit,Entire home/apt,2,NaN,NaN
10,Entire rental unit,Entire home/apt,6,2.0,NaN
13,Private room in rental unit,Private room,1,NaN,NaN
14,Private room in rental unit,Private room,1,NaN,NaN
25,Private room in rental unit,Private room,1,NaN,NaN
26,Entire condo,Entire home/apt,2,1.0,NaN
27,Entire condo,Entire home/apt,4,2.0,NaN
30,Private room in rental unit,Private room,1,NaN,NaN
36,Entire rental unit,Entire home/apt,2,1.0,NaN


In [14]:
# ============================================================
# Verify missing values in review_scores_rating
# ============================================================

listings["review_scores_rating"].isna().sum()

np.int64(0)

In [15]:
# ============================================================
# Inspect the Superhost column
# ============================================================

listings["host_is_superhost"].value_counts(dropna=False)

host_is_superhost
f      22814
t       7096
NaN      349
Name: count, dtype: int64

In [16]:
# ============================================================
# Fill missing Superhost values
# ============================================================

# Replace missing values with "f" (not a Superhost)
listings["host_is_superhost"] = listings["host_is_superhost"].fillna("f")

# Verify there are no missing values remaining
listings["host_is_superhost"].isna().sum()

np.int64(0)

In [17]:
# ============================================================
# Remove listings with missing prices
# ============================================================

# Listings without a price cannot be used for analysis.
listings = listings.dropna(subset=["price_clean"])

# Verify there are no missing prices remaining.
listings["price_clean"].isna().sum()

np.int64(0)

In [18]:
# ============================================================
# Fill missing bedroom and bed counts
# ============================================================

listings["bedrooms"] = listings["bedrooms"].fillna(0)
listings["beds"] = listings["beds"].fillna(0)

# Verify there are no missing values remaining.
listings[["bedrooms", "beds"]].isna().sum()

bedrooms    0
beds        0
dtype: int64

In [19]:
# ============================================================
# Final missing value check
# ============================================================

listings[important_columns].isna().sum()

price_clean                 0
room_type                   0
property_type               0
accommodates                0
bedrooms                    0
beds                        0
bathrooms_text            151
review_scores_rating        0
number_of_reviews           0
availability_365            0
host_is_superhost           0
neighbourhood_cleansed      0
dtype: int64

In [20]:
# ============================================================
# Inspect missing bathroom values
# ============================================================

listings.loc[
    listings["bathrooms_text"].isna(),
    ["property_type", "room_type", "accommodates", "bathrooms_text"]
].head(20)

,property_type,room_type,accommodates,bathrooms_text
141,Private room in rental unit,Private room,1,NaN
10940,Private room in home,Private room,2,NaN
11706,Private room in hostel,Private room,1,NaN
11708,Private room in hostel,Private room,1,NaN
11754,Private room in hostel,Private room,1,NaN
13675,Private room in rental unit,Private room,2,NaN
13680,Private room in rental unit,Private room,2,NaN
13681,Private room in rental unit,Private room,2,NaN
13683,Private room in rental unit,Private room,2,NaN
16599,Private room in home,Private room,4,NaN


In [21]:
# ============================================================
# Remove listings with missing bathroom information
# ============================================================

# Listings without bathroom information are a very small
# fraction of the dataset, so remove them.

listings = listings.dropna(subset=["bathrooms_text"])

# Verify there are no missing values remaining.
listings["bathrooms_text"].isna().sum()

np.int64(0)

In [22]:
listings[important_columns].isna().sum()

price_clean               0
room_type                 0
property_type             0
accommodates              0
bedrooms                  0
beds                      0
bathrooms_text            0
review_scores_rating      0
number_of_reviews         0
availability_365          0
host_is_superhost         0
neighbourhood_cleansed    0
dtype: int64

In [23]:
# ============================================================
# Final dataset shape
# ============================================================

print("Final dataset shape:", listings.shape)

Final dataset shape: (21364, 92)


In [24]:
# Save the cleaned dataset
listings.to_csv("../data/listings_clean.csv", index=False)

print("Cleaned dataset saved!")

Cleaned dataset saved!
